# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Results Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

results_schema = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("race_name", StringType(), True),
    StructField("circuit_id", StringType(), False),
    StructField("number", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("position_text", StringType(), True),
    StructField("points", IntegerType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("code", StringType(), True),
    StructField("given_name", StringType(), True),
    StructField("family_name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("constructor_id", StringType(), True),
    StructField("constructor_name", StringType(), True),
    StructField("time_millis", IntegerType(), True),
    StructField("time_gap", StringType(), True),
    StructField("fastest_lap_rank", IntegerType(), True),
    StructField("fastest_lap_number", IntegerType(), True),
    StructField("fastest_lap_time", StringType(), True),
    StructField("fastest_lap_speed_units", StringType(), True),
    StructField("fastest_lap_speed", DoubleType(), True),
])

results_input_path = f"{processed_folder_path}/results/csv/results.csv"

results_df = spark.read \
    .option("header", True) \
    .schema(results_schema) \
    .csv(results_input_path)


# 3) Transform Results Data:

The steps included:

- Drop column "code", "given_name", "family_name", "nationality", "constructor_name".
- Create Surrogate Key.
- Add Data Source and File Date.
- Fill Null Numeric cells in these columns: "time_millis", "fastest_lap_rank", "fastest_lap_number", "fastest_lap_speed" with 0.
- Fill Null cells with "None" for String type.

In [0]:
from pyspark.sql.functions import lit

results_with_audit_df = results_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

results_date_df = add_ingestion_date(results_with_audit_df)

numeric_columns = ["time_millis", "fastest_lap_rank", "fastest_lap_number", "fastest_lap_speed"]
results_numeric_filled_df = results_date_df.fillna(0, subset=numeric_columns)
results_fill_df = results_numeric_filled_df.fillna("None")

results_dropped_df = results_fill_df.drop("code", "given_name", "family_name", "nationality", "constructor_name")

results_final_df = add_surrogate_key(
    results_dropped_df,
    key_column_name="results_sk",
    hash_columns=["season", "round", "race_name", "circuit_id", "number", "position", "position_text", "points", "grid", "laps", "status", "driver_id", "constructor_id", "time_millis", "time_gap", "fastest_lap_rank", "fastest_lap_number", "fastest_lap_time", "fastest_lap_speed_units", "fastest_lap_speed"],
)

print("Final columns going into the write:", results_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
results_output_path = f"{processed_folder_path}/results/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed_for_seasons(
    input_df=results_final_df,
    db_name="f1_processed",
    table_name="results",
    output_path=results_output_path,
    merge_key_columns=["season", "round"],
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(results_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/results/delta",
    presentation_directory=f"{presentation_folder_path}/fact_results/delta",
    db_name="f1_presentation",
    table_name="fact_results",
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_results/delta"))

# 5) Save backup Results in CSV format:

In [0]:
import io
import csv

results_backup_path = f"{presentation_folder_path}/fact_results/csv/fact_results.csv"

backup_rows = [row.asDict() for row in results_final_df.collect()]
backup_fieldnames = results_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(results_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {results_backup_path}")